<p style="text-align:center"> 
    <a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/" target="_blank"> 
    <img src="../assets/logo.png" width="200" alt="Flavio Aguirre Logo"> 
    </a>
</p>

<h1 align="center"><font size="7"><strong>📉 ByeBye Predictor</strong></font></h1>
<br>
<hr>

## Telco Customer Churn – Feature Engineering

**Date:** September 17, 2025  
**Subject:** Feature Engineering & Modeling – Telco Customer Churn

In the previous notebook, we established a **baseline model** for the Telco Customer Churn dataset. The best-performing model was a **Logistic Regression**, achieving:

- AUC ≈ **0.84**
- Accuracy ≈ **0.74**
- Recall for the churn class ≈ **0.78**
- Precision for the churn class ≈ **0.51**

This baseline confirmed that the dataset contains meaningful predictive signal, but also highlighted a relevant business limitation:  
the model tends to **overestimate churners**, producing many **false positives** (customers flagged as at risk who actually stay).

In this notebook, we move beyond the baseline and focus on **feature engineering**:

- We start from the **preprocessed Telco dataset**, where all input variables are already cleaned, encoded and scaled.
- We design **domain-driven features** that capture contractual risk, lack of support, tenure buckets and interaction patterns.
- We then compare **Logistic Regression** (as a reproducible baseline) against **XGBoost**, using the feature-engineered dataset.
- Finally, we evaluate the impact of feature engineering on performance and on the churn/no-churn discrimination.

---

### Expected Final Result of Notebook 05

This notebook is designed to be a **reproducible, professional experiment** that:

1. **Implements a complete feature engineering stage** on the curated Telco dataset (already preprocessed and encoded).
2. **Builds and compares models** (Logistic Regression baseline and XGBoost) on top of these engineered features.
3. **Applies a hybrid feature selection strategy** to reduce dimensionality while preserving performance.
4. **Evaluates the models** on a held-out test set and documents the impact of feature engineering relative to the baseline model.

The outcome of this notebook will inform which model and feature set should be promoted to “production candidate” for deployment via an API or web application.

---

### Imports and setup

In [1]:
# NOTE: This notebook is intended to be run from the project root using Jupyter.

from notebooks_setup import PROJECT_ROOT

# -------- Local imports
from src.utils import get_logger
from src.data_loader import load_csv, preview_df
from src.eda import dataframe_overview
from src.feature_engineering import (
    FeatureSelectorTransformer,
    CustomCombinationTransformer,
    apply_feature_engineering,
)
from src.model_builder import ModelBuilder
from src.model_evaluation import ModelEvaluatorConfig, ModelEvaluator

# -------- External imports
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.linear_model import LogisticRegression

%matplotlib inline

2025-11-28 10:05:36,321 | utils | INFO | Added project root to sys.path: C:\Users\Pc\Desktop\github
2025-11-28 10:05:38,101 | preprocess | INFO | NLTK resource 'punkt' is already downloaded.
2025-11-28 10:05:38,103 | preprocess | INFO | NLTK resource 'stopwords' is already downloaded.
2025-11-28 10:05:38,107 | preprocess | WARNING | NLTK resource 'wordnet' not found. Downloading...
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Pc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
2025-11-28 10:05:38,227 | preprocess | INFO | NLTK resource 'punkt_tab' is already downloaded.
2025-11-28 10:05:38,228 | preprocess | INFO | Preprocess module loaded correctly and ready to be used.
2025-11-28 10:05:38,291 | src.model_builder | INFO | ModelBuilder Module initialized successfully!
2025-11-28 10:05:38,297 | src.model_evaluation | INFO | ModelEvaluator Module initialized successfully!


<br>

### Initial configuration and dataset loading

In [2]:
logger = get_logger(__name__)

# Configuration for model evaluation and reporting
config = ModelEvaluatorConfig(
    out_dir="./reports/figures/feature-engineering-model-telco/"
)

logger.info("Configuration applied:")
logger.info(f"Output directory: {config.out_dir}")
logger.info(f"Plotting style: {config.style}")

# Paths to the processed feature matrix and target vector
path_df_processed = "data/processed/telco_churn_curated.csv"
path_target_processed = "data/processed/target_processed.csv"

# Load the processed DataFrame and target variable
df_processed = load_csv(path_df_processed)
target_processed = load_csv(path_target_processed)

logger.info(f"Processed features shape: {df_processed.shape}")
preview_df(df_processed)

logger.info(f"Processed target shape: {target_processed.shape}")
preview_df(target_processed)

2025-11-28 10:05:38,321 | __main__ | INFO | Configuration applied:
2025-11-28 10:05:38,325 | __main__ | INFO | Output directory: ./reports/figures/feature-engineering-model-telco/
2025-11-28 10:05:38,333 | __main__ | INFO | Plotting style: whitegrid
2025-11-28 10:05:38,361 | src.data_loader | INFO | Loaded CSV: data/processed/telco_churn_curated.csv | Shape: (7043, 30)
2025-11-28 10:05:38,363 | src.data_loader | INFO | Completed: load_csv
2025-11-28 10:05:38,367 | src.data_loader | INFO | Loaded CSV: data/processed/target_processed.csv | Shape: (7043, 1)
2025-11-28 10:05:38,370 | src.data_loader | INFO | Completed: load_csv
2025-11-28 10:05:38,370 | __main__ | INFO | Processed features shape: (7043, 30)
2025-11-28 10:05:38,372 | __main__ | INFO | Processed target shape: (7043, 1)


,churn
0,0
1,0
2,1
3,0
4,1


### Splitting data

#### Train/Test Split

Before applying any feature engineering, we split the dataset into **train** and **test** subsets.

This is critical to avoid **data leakage**: feature engineering and feature selection will be learned **only on the training data** and later applied to the test set, which remains unseen during training.

#### Train/test split with index sanity check

In [3]:
# Define feature matrix and target vector
X = df_processed
y = target_processed.squeeze()

# Sanity check: ensure indices align
if (X.index == y.index).all():
    logger.info("Indices of X and y match correctly.")
else:
    raise ValueError("Indices of X and y do NOT match.")

RND = 42
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RND,
    stratify=y,
)

logger.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

2025-11-28 10:05:38,397 | __main__ | INFO | Indices of X and y match correctly.
2025-11-28 10:05:38,407 | __main__ | INFO | Train shape: (5634, 30), Test shape: (1409, 30)


## Feature Engineering Strategy

In this section, we design and apply feature engineering on top of the **already preprocessed** Telco dataset.

The guiding principles are:

1. **Numeric derivatives and ratios**  
   - Example: `ratio_total_monthly` to relate total charges to monthly charges.

2. **Commercially meaningful tenure buckets**  
   - Group customers into segments (e.g. new, early, established, loyal) based on their tenure.

3. **Interaction flags**  
   - Capture risky combinations such as:
     - Fiber optic with no online security.
     - Paperless billing combined with electronic check.
     - Senior citizens paying higher-than-median monthly charges.

4. **Aggregated “risk” scores**  
   - Example: `no_protection_score`, summarizing the absence of protective services (security, device protection, tech support).

5. **Hybrid feature selection**  
   - Start with a filter method (`SelectKBest` with mutual information).
   - Refine using a wrapper method (`RFECV` with Logistic Regression) to remove redundant or weakly informative features.

All transformations are implemented as **custom transformers**, integrated into the existing feature engineering framework in `src/feature_engineering.py`.

#### Custom feature transformation functions

In [4]:
def add_ratio_total_monthly(X: pd.DataFrame) -> pd.DataFrame:
    """
    Create a ratio between total and monthly charges.
    Intuition:
    - High ratio can capture customers with long tenure and accumulated cost.
    - Low ratio can indicate new customers or inconsistent billing.
    """
    X = X.copy()
    X["ratio_total_monthly"] = np.where(
        X["monthly_charges"] == 0,
        0,
        X["total_charges"] / (X["monthly_charges"] + 1e-9),
    )
    return X[["ratio_total_monthly"]]


def tenure_bins(X: pd.DataFrame) -> pd.DataFrame:
    """
    Bucket tenure into commercial segments:
    - 0-6 months:     new customers
    - 7-24 months:    early stage
    - 25-60 months:   established
    - >60 months:     loyal
    """
    X = X.copy()
    bins = [-1, 6, 24, 60, 9999]
    labels = [0, 1, 2, 3]
    X["tenure_bucket"] = pd.cut(X["tenure"], bins=bins, labels=labels)

    # Handle missing as a separate category, then cast to int
    X["tenure_bucket"] = X["tenure_bucket"].cat.add_categories([-1]).fillna(-1).astype(int)

    dummies = pd.get_dummies(X["tenure_bucket"], prefix="tenure_bkt")

    # Ensure consistent columns even if some buckets are missing in a given split
    expected_cols = [f"tenure_bkt_{i}" for i in range(4)]
    dummies = dummies.reindex(columns=expected_cols, fill_value=0)

    return dummies


def interaction_flags(X: pd.DataFrame) -> pd.DataFrame:
    """
    Build interaction flags that capture risky combinations:
    - fiber_and_no_security: fiber optic customers without online security.
    - paperless_and_echeck: paperless billing + electronic check payments.
    - senior_and_single_monthly_high: senior citizens with above-median monthly charges.
    """
    X = X.copy()
    X["fiber_and_no_security"] = (
        (X["internet_service_fiber_optic"] == 1) & (X["online_security_yes"] == 0)
    ).astype(int)

    X["paperless_and_echeck"] = (
        (X["paperless_billing_yes"] == 1)
        & (X["payment_method_electronic_check"] == 1)
    ).astype(int)

    X["senior_and_single_monthly_high"] = (
        (X["senior_citizen"] == 1)
        & (X["monthly_charges"] > X["monthly_charges"].median())
    ).astype(int)

    return X[
        [
            "fiber_and_no_security",
            "paperless_and_echeck",
            "senior_and_single_monthly_high",
        ]
    ]


def no_protection_score(X: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate absence of protective services into a single score.
    The score is normalized to [0, 1]:
    - 0 means all protections are active.
    - 1 means no protection at all.
    """
    X = X.copy()
    flags = (
        (X["online_security_yes"] == 0).astype(int)
        + (X["device_protection_yes"] == 0).astype(int)
        + (X["tech_support_yes"] == 0).astype(int)
    )
    X["no_protection_score"] = flags / 3.0
    return X[["no_protection_score"]]

#### Wrap custom functions into transformers and apply FE

In [5]:
try:
    ratio_transformer = CustomCombinationTransformer(
        add_ratio_total_monthly,
        ["ratio_total_monthly"],
    )
    tenure_transformer = CustomCombinationTransformer(
        tenure_bins,
        ["tenure_bkt_0", "tenure_bkt_1", "tenure_bkt_2", "tenure_bkt_3"],
    )
    interaction_transformer = CustomCombinationTransformer(
        interaction_flags,
        ["fiber_and_no_security", "paperless_and_echeck", "senior_and_single_monthly_high"],
    )
    no_prot_transformer = CustomCombinationTransformer(
        no_protection_score,
        ["no_protection_score"],
    )

    logger.info("Custom transformers instantiated successfully.")

except Exception as e:
    logger.error(f"Error while instantiating CustomCombinationTransformer: {e}")
    raise

transformers = [
    ("ratio", ratio_transformer),
    ("tenure", tenure_transformer),
    ("interaction", interaction_transformer),
    ("no_prot", no_prot_transformer),
]

# Apply feature engineering on train and test separately (to avoid data leakage)
X_train_feats = apply_feature_engineering(X_train, transformers)
X_test_feats = apply_feature_engineering(X_test, transformers)

logger.info(f"X_train with engineered features: {X_train_feats.shape}")
preview_df(X_train_feats)

logger.info(f"X_test with engineered features: {X_test_feats.shape}")
preview_df(X_test_feats)

# Check for NaNs in the new feature space
logger.info(
    f"NaNs in engineered train features: {X_train_feats.isna().sum().sum()}"
)
logger.info(
    f"NaNs in engineered test features: {X_test_feats.isna().sum().sum()}"
)

2025-11-28 10:05:38,444 | __main__ | INFO | Custom transformers instantiated successfully.
2025-11-28 10:05:38,447 | src.feature_engineering | INFO | Applied transformer 'ratio', new shape: (5634, 31)
2025-11-28 10:05:38,447 | src.feature_engineering | INFO | Applied transformer 'tenure', new shape: (5634, 35)
2025-11-28 10:05:38,461 | src.feature_engineering | INFO | Applied transformer 'interaction', new shape: (5634, 38)
2025-11-28 10:05:38,465 | src.feature_engineering | INFO | Applied transformer 'no_prot', new shape: (5634, 39)
2025-11-28 10:05:38,467 | src.feature_engineering | INFO | Applied transformer 'ratio', new shape: (1409, 31)
2025-11-28 10:05:38,469 | src.feature_engineering | INFO | Applied transformer 'tenure', new shape: (1409, 35)
2025-11-28 10:05:38,472 | src.feature_engineering | INFO | Applied transformer 'interaction', new shape: (1409, 38)
2025-11-28 10:05:38,477 | src.feature_engineering | INFO | Applied transformer 'no_prot', new shape: (1409, 39)
2025-11-28 

## Hybrid Feature Selection

Before comparing models, we apply a **hybrid feature selection strategy**:

1. **Filter step** – `SelectKBest` with `mutual_info_classif`  
   - Removes the weakest features based on their relationship with the target.

2. **Wrapper step** – `RFECV` with Logistic Regression  
   - Recursively eliminates features while optimizing a cross-validated **F1-score**.

This approach balances **speed** (via the filter) and **robustness** (via the wrapper), and helps us avoid overfitting by removing noisy or redundant features.

#### Feature selection with FeatureSelectorTransformer

In [6]:
# Step 1: Define the hybrid selector pipeline
hybrid_selector = Pipeline(
    [
        ("filter", SelectKBest(score_func=mutual_info_classif, k=39)),
        (
            "wrapper",
            RFECV(
                estimator=LogisticRegression(max_iter=1000, solver="liblinear"),
                step=1,
                cv=7,
                scoring="f1",
            ),
        ),
    ]
)

# Step 2: Wrap in the FeatureSelectorTransformer
fe_selector = FeatureSelectorTransformer(feature_selector=hybrid_selector)

# Step 3: Fit on the training set and transform both train and test
X_train_selected = fe_selector.fit_transform(X_train_feats, y_train)
X_test_selected = fe_selector.transform(X_test_feats)

logger.info(f"Original feature space: {X_train_feats.shape}")
logger.info(f"Selected feature space: {X_train_selected.shape}")
logger.info(f"Selected features: {fe_selector._feature_names_out}")

2025-11-28 10:05:42,136 | __main__ | INFO | Original feature space: (5634, 39)
2025-11-28 10:05:42,137 | __main__ | INFO | Selected feature space: (5634, 35)
2025-11-28 10:05:42,137 | __main__ | INFO | Selected features: ['monthly_charges', 'tenure', 'total_charges', 'gender_male', 'partner_yes', 'dependents_yes', 'phone_service_yes', 'multiple_lines_no_phone_service', 'multiple_lines_yes', 'internet_service_fiber_optic', 'internet_service_no', 'online_security_no_internet_service', 'online_security_yes', 'online_backup_no_internet_service', 'online_backup_yes', 'device_protection_no_internet_service', 'device_protection_yes', 'tech_support_no_internet_service', 'tech_support_yes', 'streaming_tv_no_internet_service', 'streaming_tv_yes', 'streaming_movies_no_internet_service', 'streaming_movies_yes', 'contract_one_year', 'contract_two_year', 'paperless_billing_yes', 'payment_method_credit_card_automatic', 'payment_method_electronic_check', 'payment_method_mailed_check', 'senior_citizen'

### Model Comparison on Engineered Features

With the engineered and selected features in place, we now compare:

- **Logistic Regression** (baseline-friendly, interpretable).
- **XGBoost** (gradient-boosted trees, typically stronger on tabular data).

We reuse the `ModelBuilder` class, but this time we feed it with `X_train_selected`.  
Cross-validation is used to estimate average performance across folds.

#### Model evaluation with ModelBuilder

In [7]:
# 1) Instantiate ModelBuilder
mb = ModelBuilder(task="classification", random_state=RND)

# 2) Evaluate models using cross-validation on the engineered + selected features
models_evaluation_fe = mb.evaluate_models(X_train_selected, y_train, cv=7)
evaluation_results_df_fe = pd.DataFrame(models_evaluation_fe)

logger.info("Cross-validated results (7-fold average) after feature engineering + selection:")
display(evaluation_results_df_fe)

# 3) Select the best model according to test F1-score (from CV results)
best_model_fe = mb.select_best_model(metric="test_f1")
best_model_name_fe = mb.best_model_name_

logger.info(f"Best model after FE + selection according to F1: '{best_model_name_fe}'")

# 4) Train that specific model on the full training set (engineered + selected features)
final_model_fe = mb.train_final_model(
    best_model_name_fe,
    X_train_selected,
    y_train,
)

logger.info(
    f"Final model '{best_model_name_fe}' trained with {X_train_selected.shape[0]} samples "
    f"and {X_train_selected.shape[1]} selected features."
)

2025-11-28 10:05:42,158 | src.model_builder | INFO | Cross-validating model: logistic_regression...
2025-11-28 10:05:44,945 | src.model_builder | INFO | Model logistic_regression evaluated successfully.
2025-11-28 10:05:44,947 | src.model_builder | INFO | Cross-validating model: decision_tree...
2025-11-28 10:05:46,751 | src.model_builder | INFO | Model decision_tree evaluated successfully.
2025-11-28 10:05:46,752 | src.model_builder | INFO | Cross-validating model: random_forest...
2025-11-28 10:05:47,583 | src.model_builder | INFO | Model random_forest evaluated successfully.
2025-11-28 10:05:47,583 | src.model_builder | INFO | Cross-validating model: xgboost...
2025-11-28 10:05:47,901 | src.model_builder | INFO | Model xgboost evaluated successfully.
2025-11-28 10:05:47,903 | src.data_loader | INFO | Completed: evaluate_models
2025-11-28 10:05:47,905 | __main__ | INFO | Cross-validated results (7-fold average) after feature engineering + selection:


,logistic_regression,decision_tree,random_forest,xgboost
fit_time,0.051170,0.031985,0.566617,0.127668
score_time,0.017534,0.011064,0.043497,0.016682
test_accuracy,0.753807,0.730915,0.790019,0.782393
test_precision,0.524591,0.493259,0.637215,0.606869
test_recall,0.795994,0.517052,0.490335,0.511082
test_f1,0.632180,0.504703,0.553728,0.554659
test_roc_auc,0.847300,0.663097,0.821829,0.824966


2025-11-28 10:05:47,913 | src.model_builder | INFO | Best model selected: logistic_regression with test_f1: 0.6322
2025-11-28 10:05:47,914 | src.data_loader | INFO | Completed: select_best_model
2025-11-28 10:05:47,915 | __main__ | INFO | Best model after FE + selection according to F1: 'logistic_regression'
2025-11-28 10:05:47,916 | src.model_builder | INFO | Training final model: logistic_regression...
2025-11-28 10:05:47,953 | src.model_builder | INFO | Model logistic_regression trained successfully.
2025-11-28 10:05:47,954 | src.data_loader | INFO | Completed: train_final_model
2025-11-28 10:05:47,955 | __main__ | INFO | Final model 'logistic_regression' trained with 5634 samples and 35 selected features.


<br>

## Final Evaluation on the Test Set

We now evaluate the **best model after feature engineering and selection** on the **held-out test set**:

- Input: `X_test_selected`
- Target: `y_test`

We use the same `ModelEvaluator` infrastructure as in the baseline notebook, to allow **direct, apples-to-apples comparison**:

- Classification metrics (precision, recall, F1).
- Confusion matrix.
- ROC curve.
- Precision-Recall curve.

#### Test evaluation with ModelEvaluator

In [8]:
evaluator_fe = ModelEvaluator(task="classification", save_dir=config.out_dir)

# Predictions and probabilities on the test set
y_pred_fe = final_model_fe.predict(X_test_selected)
y_proba_fe = final_model_fe.predict_proba(X_test_selected)[:, 1]

# Run evaluation
metrics_fe = evaluator_fe.evaluate(
    final_model_fe,
    X_test_selected,
    y_test,
    run_name="feature_engineering_telco",
)

logger.info("Final test metrics after feature engineering and selection:")
logger.info(metrics_fe)

2025-11-28 10:05:47,982 | src.data_loader | INFO | Completed: evaluate
2025-11-28 10:05:47,985 | __main__ | INFO | Final test metrics after feature engineering and selection:
2025-11-28 10:05:47,988 | __main__ | INFO | {'accuracy': 0.7437899219304471, 'precision': 0.8019405005197715, 'recall': 0.7437899219304471, 'f1': 0.7573232989454067}


### Plots (confusion matrix, ROC curve, Precision-Recall curve)

In [9]:
# Confusion matrix
confusion_matrix_fe = evaluator_fe.plot_confusion_matrix(
    y_test,
    y_pred_fe,
    labels=["No Churn", "Churn"],
    title=f"Feature-engineered model confusion matrix ({best_model_name_fe})",
    filename=f"confusion_matrix_feature_engineering_{best_model_name_fe}.png",
)

# ROC curve
roc_curve_fe = evaluator_fe.plot_roc_curve(
    final_model_fe,
    X_test_selected,
    y_test,
    title=f"Feature-engineered ROC Curve ({best_model_name_fe})",
    filename=f"roc_curve_feature_engineering_{best_model_name_fe}.png",
)

# Precision-Recall curve
precision_recall_fe = evaluator_fe.plot_precision_recall(
    final_model_fe,
    X_test_selected,
    y_test,
    title=f"Feature-engineered Precision-Recall Curve ({best_model_name_fe})",
    filename=f"precision_recall_curve_feature_engineering_{best_model_name_fe}.png",
)

2025-11-28 10:05:48,158 | src.model_evaluation | INFO | Plot saved to ./reports/figures/feature-engineering-model-telco/confusion_matrix_feature_engineering_logistic_regression.png
2025-11-28 10:05:48,270 | src.model_evaluation | INFO | Plot saved to ./reports/figures/feature-engineering-model-telco/roc_curve_feature_engineering_logistic_regression.png
2025-11-28 10:05:48,387 | src.model_evaluation | INFO | Plot saved to ./reports/figures/feature-engineering-model-telco/precision_recall_curve_feature_engineering_logistic_regression.png


### Saving artifacts

To reuse this experiment in future stages (batch scoring, API, model comparison), we persist:

- The final feature-engineered model (`final_model_fe`), using the official `ModelBuilder.save_model` method.
- The feature selector (`fe_selector`), using `joblib`.

This allows us to later reload both artifacts and apply the same feature engineering and selection steps to new data.

In [10]:
# Directory where the feature-engineered models will be stored
model_dir = "models/feature_engineering"
os.makedirs(model_dir, exist_ok=True)

# Filepaths for the model and the feature selector
model_path = os.path.join(
    model_dir,
    f"{best_model_name_fe}_fe_v1_2025-09-17.joblib"
)
selector_path = os.path.join(
    model_dir,
    "feature_selector_fe_v1_2025-11-28.joblib"
)

# 1) Save the trained feature-engineered model using ModelBuilder
mb.save_model(best_model_name_fe, model_path)

# 2) Save the feature selector so we can reproduce the selected feature space
joblib.dump(fe_selector, selector_path)
logger.info(f"Feature selector saved to: {selector_path}")

2025-11-28 10:05:48,405 | src.model_builder | INFO | Model logistic_regression saved to models/feature_engineering\logistic_regression_fe_v1_2025-09-17.joblib
2025-11-28 10:05:48,407 | src.data_loader | INFO | Completed: save_model
2025-11-28 10:05:48,411 | __main__ | INFO | Feature selector saved to: models/feature_engineering\feature_selector_fe_v1_2025-11-28.joblib


## Results and Insights

### Feature-engineered best model

From the evaluation log we obtain, for the best feature-engineered model (`logistic_regression`):

- Accuracy ≈ **0.74**
- Precision (overall) ≈ **0.80**
- Recall (overall) ≈ **0.74**
- F1 (overall) ≈ **0.76**


### Comparison with the Baseline

From the baseline notebook, we know that:

- **Baseline Logistic Regression (no feature engineering)**
  - AUC ≈ 0.84
  - Accuracy ≈ 0.74
  - Recall (Churn) ≈ 0.78
  - Precision (Churn) ≈ 0.51

After applying **feature engineering + hybrid feature selection**, and retraining the models on the engineered feature space, we observe:

- The **best-performing model** (according to cross-validated F1-score) is:  
  `logistic_regression` (as selected by `ModelBuilder`).
- On the test set, the feature-engineered model:
  - Preserves the overall **discrimination** between churners and non-churners (accuracy ~0.74, F1 ~0.76).
  - Achieves a **much higher overall precision** (~0.80), which is valuable from a business perspective when the cost of false positives is high.
  - Encodes business-relevant risk signals, such as:
    - Customers with **fiber optic service but no online security**.
    - Customers without any protection services (`no_protection_score` close to 1).
    - Senior customers with consistently high monthly charges.
    - Tenure segments with higher propensity to churn.

> Even when the raw metric gains are modest, the **quality of the features** matters from a business perspective:
> - It becomes easier to explain **why** a segment is at risk (e.g., no support, high charges, vulnerable contract combinations).
> - Retention teams can design **targeted interventions** around these signals.

### Business Interpretation of Engineered Features

- **`ratio_total_monthly`**  
  Highlights customers with a long charging history relative to their monthly fee. This can detect long-tenure customers who may feel “overcharged” over time.

- **`tenure_bkt_*` (0–3)**  
  Encodes acquisition vs consolidation stages. Early-tenure buckets are traditionally more volatile, while loyal customers show different churn dynamics.

- **`fiber_and_no_security`**  
  Flags high-tech customers who are not protected. This combination may correlate with frustration around service quality or perceived risk.

- **`paperless_and_echeck`**  
  Combines payment and billing preferences that may interact with perceived billing transparency.

- **`no_protection_score`**  
  Aggregates missing support/protection services into a continuous score.  
  Customers with higher scores are more exposed and potentially more frustrated.

---

## Conclusion of this Notebook

- We designed and implemented a **feature engineering pipeline** on top of the curated Telco dataset.
- We constructed **domain-informed features** that capture tenure, protection, and interaction risks.
- We applied a **hybrid feature selection** approach to retain only the most informative features.
- We compared **Logistic Regression and XGBoost** on the engineered feature space using cross-validation.
- We evaluated the **best feature-engineered model** on a hold-out test set using the same evaluation framework as the baseline.

This notebook concludes the **feature engineering and model evaluation phase** for the Telco dataset and provides a solid experimental basis for:

1. **Hyperparameter tuning** of the chosen model.
2. **Integration of textual features** (e.g., sentiment from Reddit comments) into a hybrid model.
3. **Selection of a production-ready model** to be serialized and exposed via an API or simple web application.

---

### Author

<a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/">**Flavio Aguirre**</a><br>
<a href="https://coursera.org/share/e27ae5af81b56f99a2aa85289b7cdd04">***Data Scientist***</a>